In [ ]:
import re
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [10]:
DATA_DIR = Path("./prepared_data")
OUT_DIR = Path("./prepared_data_clean")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "train": DATA_DIR / "train.jsonl",
    "valid": DATA_DIR / "valid.jsonl",
    "test_clean": DATA_DIR / "test_clean.jsonl",
    "test_noisy_easy": DATA_DIR / "test_noisy_easy.jsonl",
    "test_noisy_hard": DATA_DIR / "test_noisy_hard.jsonl",
}

In [11]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"JSON error in {path} at line {i}: {e}")
    return records


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [12]:
datasets = {}

for split, path in FILES.items():
    if path.exists():
        datasets[split] = load_jsonl(path)
        print(f"{split}: {len(datasets[split])} samples")
    else:
        print(f"{split}: file not found -> {path}")

train: 263547 samples
valid: 6436 samples
test_clean: 6440 samples
test_noisy_easy: 1920 samples
test_noisy_hard: 1980 samples


In [13]:
required_fields = {"input_text", "target_text"}

for split, records in datasets.items():
    bad = 0
    for i, rec in enumerate(records):
        missing = required_fields - set(rec.keys())
        if missing:
            bad += 1
            print(f"{split} | sample {i} missing: {missing}")
    print(f"{split}: missing-field samples = {bad}")

train: missing-field samples = 0
valid: missing-field samples = 0
test_clean: missing-field samples = 0
test_noisy_easy: missing-field samples = 0
test_noisy_hard: missing-field samples = 0


In [14]:
def clean_text(text:str)-> str:
    if not isinstance(text,str):
        return ''
    text= text.replace("\n","")
    text= text.replace("\t","")

    # remove citation tags like @xcite
    text = re.sub(r"@xcite", " ", text)

    # remove multiple underscores around words: _ word _ -> word
    text = re.sub(r"\b_\s*", "", text)
    text = re.sub(r"\s*_\b", "", text)

    # remove page-number-like stray digits at end
    text = re.sub(r"\s+\d+\s*$", "", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [15]:
cleaned_datasets = {}

for split, records in datasets.items():
    cleaned = []
    for rec in tqdm(records, desc=f"Cleaning {split}"):
        new_rec = rec.copy()
        new_rec["input_text"] = clean_text(rec.get("input_text", ""))
        new_rec["target_text"] = clean_text(rec.get("target_text", ""))
        cleaned.append(new_rec)
    cleaned_datasets[split] = cleaned

print("Done cleaning.")

Cleaning test_noisy_hard: 100%|██████████| 1980/1980 [00:00<00:00, 4148.00it/s]

Done cleaning.


In [16]:
split = "train"
idx = 0

print("BEFORE:\n")
print(datasets[split][idx]["input_text"][:1000])
print("\n" + "="*80 + "\n")
print("AFTER:\n")
print(cleaned_datasets[split][idx]["input_text"][:1000])

BEFORE:

summarize: question: Summarize the main contribution of this paper context: additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models . many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.g. @xcite . in the last years many interesting results on learning rates of regularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used , see e.g. @xcite , @xcite

In [17]:
def text_len_words(text):
    return len(text.split())

length_stats = []

for split, records in cleaned_datasets.items():
    for rec in records:
        length_stats.append({
            "split": split,
            "input_words": text_len_words(rec["input_text"]),
            "target_words": text_len_words(rec["target_text"]),
        })

df_len = pd.DataFrame(length_stats)
df_len.groupby("split")[["input_words", "target_words"]].describe()

input_words                                               \
                      count        mean         std    min    25%    50%   
split                                                                      
test_clean           6440.0  186.744720   31.004447   36.0  172.0  183.0   
test_noisy_easy      1920.0  510.335417   44.829304  346.0  484.0  510.0   
test_noisy_hard      1980.0  512.727778   38.573321  369.0  490.0  511.0   
train              263547.0  255.542343  141.329994   10.0  170.0  186.0   
valid                6436.0  186.140615   29.610409   13.0  171.0  184.0   

                              target_words                                \
                   75%    max        count        mean         std   min   
split                                                                      
test_clean       196.0  595.0       6440.0  162.526087   57.539145  44.0   
test_noisy_easy  534.0  774.0       1920.0  163.234375   57.057163  44.0   
test_noisy_hard  532.0  766.0       1980.0  164.110606   58.356914  46.0   
train            284.0  800.0     263547.0  274.702584  577.357517   2.0   
valid            196.0  452.0       6436.0  160.931479   58.196074  44.0   

                                               
                   25%    50%    75%      max  
split                                          
test_clean       117.0  159.0  206.0    290.0  
test_noisy_easy  119.0  159.0  207.0    286.0  
test_noisy_hard  118.0  161.0  209.0    285.0  
train            114.0  163.0  236.0  25490.0  
valid            115.0  157.0  204.0    290.0

In [18]:
for split, records in cleaned_datasets.items():
    long_count = 0
    for rec in records:
        if text_len_words(rec["input_text"]) > 450:
            long_count += 1
    print(f"{split}: input > 450 words = {long_count}")

train: input > 450 words = 54541
valid: input > 450 words = 1
test_clean: input > 450 words = 6
test_noisy_easy: input > 450 words = 1780
test_noisy_hard: input > 450 words = 1912


In [19]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "is", "are",
    "was", "were", "be", "been", "with", "as", "by", "that", "this", "it",
    "we", "our", "their", "from", "at", "which", "these", "those", "can",
    "may", "using", "use", "used", "into", "than", "such"
}

def normalize_tokens(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    toks = [t for t in text.split() if len(t) > 2 and t not in STOPWORDS]
    return toks

def lexical_overlap_ratio(input_text, target_text):
    a = set(normalize_tokens(input_text))
    b = set(normalize_tokens(target_text))
    if len(b) == 0:
        return 0.0
    return len(a & b) / len(b)

In [20]:
for split, records in cleaned_datasets.items():
    for rec in records:
        rec["overlap_ratio"] = round(
            lexical_overlap_ratio(rec["input_text"], rec["target_text"]), 4
        )

for split, records in cleaned_datasets.items():
    ratios = [r["overlap_ratio"] for r in records]
    print(split, "avg overlap =", round(sum(ratios)/len(ratios), 4))

train avg overlap = 0.2349
valid avg overlap = 0.2263
test_clean avg overlap = 0.2242
test_noisy_easy avg overlap = 0.2853
test_noisy_hard avg overlap = 0.2868


In [21]:
split = "test_noisy_hard"

suspects = sorted(
    cleaned_datasets[split],
    key=lambda x: x["overlap_ratio"]
)[:5]

for i, rec in enumerate(suspects):
    print(f"\n===== SUSPECT {i+1} | overlap={rec['overlap_ratio']} =====")
    print("PAPER ID:", rec.get("paper_id"))
    print("\nINPUT:\n", rec["input_text"][:1200])
    print("\nTARGET:\n", rec["target_text"][:600])


===== SUSPECT 1 | overlap=0.0625 =====
PAPER ID: test_575

INPUT:
 summarize: question: Summarize the main contribution of this paper context: atoms in highly excited rydberg states ( principal quantum number @xmath3 ) have large radii and electric - dipole transition matrix elements ( @xmath4 ) , large polarizabilities ( @xmath5 ) and strong van - der - waals interactions ( @xmath6 ) .these properties have led to a variety of interesting investigations and applications , including quantum information and logic gates , single - photon sources enabled by the rydberg excitation blockade effect , and many - body physics with strong long - range interactions . the large polarizability makes rydberg atoms sensitive to external fields , giving rise to applications in field measurement , quantum control and studies involving collisions and novel molecules . ( circles ) are prepared at an electric field @xmath7 = 3.14 v / cm . the field is then linearly ramped to @xmath8 = 3.99 v since the di

In [23]:
THRESHOLDS = {
    "train": 0.03,
    "valid": 0.03,
    "test_clean": 0.03,
    "test_noisy_easy": 0.02,
    "test_noisy_hard": 0.01,
}
filtered_datasets = {}

for split, records in cleaned_datasets.items():
    th = THRESHOLDS[split]
    kept = [r for r in records if r["overlap_ratio"] >= th]
    filtered_datasets[split] = kept

    print(
        f"{split}: before={len(records)}, after={len(kept)}, removed={len(records)-len(kept)}"
    )

train: before=263547, after=260386, removed=3161
valid: before=6436, after=6424, removed=12
test_clean: before=6440, after=6430, removed=10
test_noisy_easy: before=1920, after=1919, removed=1
test_noisy_hard: before=1980, after=1980, removed=0


In [25]:
for split, records in filtered_datasets.items():
    for rec in records:
        rec.pop("overlap_ratio", None)
for split, records in filtered_datasets.items():
    out_path = OUT_DIR / f"{split}.jsonl"
    save_jsonl(records, out_path)
    print("Saved:", out_path)        

Saved: prepared_data_clean\train.jsonl
Saved: prepared_data_clean\valid.jsonl
Saved: prepared_data_clean\test_clean.jsonl
Saved: prepared_data_clean\test_noisy_easy.jsonl
Saved: prepared_data_clean\test_noisy_hard.jsonl
